This notebook trains two lightweight models to replace the RoBERTa-based teacher pipeline:

| Model | Task | Algorithm | Evaluated F1 |
|---|---|---|---|
| **Span Identifier (SI)** | Binary: does this sentence contain propaganda? | Logistic Regression + TF-IDF (word + char n-grams) | ~0.51 (PROP) |
| **Technique Classifier (TC)** | 14-class: which propaganda technique is used? | SGD / Modified-Huber SVM + TF-IDF (word + char n-grams) | ~0.38 macro F1 |

**Architecture rationale:**
- The dataset contains long news articles where propaganda spans are sparse (~7% of tokens). Predicting at the token level produces extreme class imbalance and poor generalisation.
- Splitting documents into sentences first reduces the task to a balanced binary classification problem (18% of sentences contain at least one propaganda span), which TF-IDF + LR handles effectively.
- Technique classification operates on the extracted span text only. Word + character n-grams capture both lexical patterns (specific phrases used in each technique) and morphological patterns (suffixes, prefixes) that distinguish techniques like *Loaded Language* from *Flag-Waving*.
- No neural networks, no GPU, no transformers — the full pipeline runs on CPU in under 3 minutes.

In [12]:
import ast
import re
import collections
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [15]:
BASE_DIR   = Path.cwd().resolve().parent
DATA_PATH  = BASE_DIR / 'data' / 'processed' / 'distilled.csv'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SI_PATH = MODELS_DIR / 'distilled_si.joblib'
TC_PATH = MODELS_DIR / 'distilled_tc.joblib'

RANDOM_STATE = 42
TEST_SIZE    = 0.20

print(f'Data  : {DATA_PATH}')
print(f'SI out: {SI_PATH}')
print(f'TC out: {TC_PATH}')

Data  : /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/data/processed/distilled.csv
SI out: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_si.joblib
TC out: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_tc.joblib


In [17]:
#Load and parse data
def get_technique(item: dict) -> str:
    """Extract technique label, tolerating the two malformed rows in the dataset
    where the key name matches the lowercased technique rather than 'technique'."""
    if 'technique' in item:
        return item['technique']
    for k, v in item.items():
        if k != 'span':
            return v
    return None


df = pd.read_csv(DATA_PATH)
df = df.reset_index(drop=True)
df.head()

,Unnamed: 0,text,propaganda
0,0,Outrage as Donald Trump suggests injecting dis...,"[{'span': [0, 6], 'technique': 'Loaded_Languag..."
1,1,The senator's vile betrayal of working familie...,"[{'span': [14, 18], 'technique': 'Loaded_Langu..."
2,2,Brave freedom fighters resist the tyrannical o...,"[{'span': [0, 5], 'technique': 'Loaded_Languag..."
3,3,The corrupt elites are bleeding this country d...,"[{'span': [4, 10], 'technique': 'Loaded_Langua..."
4,4,A catastrophic failure of leadership has plung...,"[{'span': [2, 13], 'technique': 'Loaded_Langua..."


In [18]:
#Parse the 'propaganda' column from string → list of dicts
df['propaganda_parsed'] = df['propaganda'].apply(ast.literal_eval)

print(f'Rows loaded : {len(df):,}')
print(f'Columns     : {df.columns.tolist()}')
print()

Rows loaded : 4,506
Columns     : ['Unnamed: 0', 'text', 'propaganda', 'propaganda_parsed']



In [20]:
#Summarise techniques
all_techniques = []
for entries in df['propaganda_parsed']:
    for item in entries:
        t = get_technique(item)
        if t:
            all_techniques.append(t)

tech_counts = collections.Counter(all_techniques)
print(f'Total span annotations : {len(all_techniques):,}')
print(f'Unique techniques      : {len(tech_counts)}')
print()
print('Technique distribution:')
for tech, cnt in tech_counts.most_common():
    print(f'  {tech:<45s} {cnt:>6,}')

Total span annotations : 23,789
Unique techniques      : 16

Technique distribution:
  Bandwagon_Reductio_ad_hitlerum                 6,083
  Loaded_Language                                2,821
  Whataboutism_Straw_Men_Red_Herring             2,744
  Flag-Waving                                    2,254
  Appeal_to_Authority                            1,849
  Doubt                                          1,541
  Name_Calling_Labeling                          1,498
  Appeal_to_fear-prejudice                       1,463
  Exaggeration_Minimisation                      1,190
  Repetition                                     1,094
  Causal_Oversimplification                        459
  Black-and-White_Fallacy                          339
  Slogans                                          260
  Thought-terminating_Cliches                      191
  Red_Herring                                        2
  Minimisation                                       1


In [21]:
#Feature engineering
def split_sentences(text: str):
    """Split text into sentences, returning (sentence_str, start_char, end_char).

    Uses a simple regex that splits on sentence-ending punctuation. This is
    intentionally lightweight — the goal is reasonable chunks, not perfect
    linguistic sentences.
    """
    sents = []
    for m in re.finditer(r'[^.!?\n]+[.!?\n]+|[^.!?\n]+$', text):
        s = m.group().strip()
        if s:
            sents.append((s, m.start(), m.end()))
    return sents

In [22]:
def build_sentence_dataset(dataframe: pd.DataFrame):
    """Build sentence-level binary dataset for Span Identification.

    Returns
    -------
    sentences : list[str]
    labels    : list[int]  — 1 if sentence contains ≥1 propaganda span, else 0
    """
    sentences, labels = [], []
    for _, row in dataframe.iterrows():
        text    = row['text']
        entries = row['propaganda_parsed']
        sents   = split_sentences(text)

        #Collect all char-level prop ranges for this document
        prop_ranges = [(item['span'][0], item['span'][1]) for item in entries]

        for sent, ss, se in sents:
            if len(sent.split()) < 3:  #skip trivially short fragments
                continue
            has_prop = any(ps < se and pe > ss for ps, pe in prop_ranges)
            sentences.append(sent)
            labels.append(1 if has_prop else 0)

    return sentences, labels

In [23]:
def build_span_dataset(dataframe: pd.DataFrame, min_count: int = 5):
    """Build span-level multi-class dataset for Technique Classification.

    Parameters
    ----------
    min_count : drop technique classes with fewer than this many examples

    Returns
    -------
    span_texts : list[str]
    techniques : list[str]
    classes    : list[str]  — sorted list of retained class names
    """
    span_texts, techniques = [], []
    for _, row in dataframe.iterrows():
        text    = row['text']
        entries = row['propaganda_parsed']
        for item in entries:
            tech = get_technique(item)
            s, e = item['span']
            span_txt = text[s:e].strip()
            if span_txt and tech:
                span_texts.append(span_txt)
                techniques.append(tech)

    #Drop rare classes
    counts = collections.Counter(techniques)
    rare   = {k for k, v in counts.items() if v < min_count}
    keep   = [(t, y) for t, y in zip(span_texts, techniques) if y not in rare]
    span_texts = [x[0] for x in keep]
    techniques = [x[1] for x in keep]
    classes    = sorted(set(techniques))

    return span_texts, techniques, classes

In [29]:
print('Building sentence-level dataset...')
sentences, y_sent = build_sentence_dataset(df)
print(f'Sentences total: {len(sentences):,}')
print(f'Prop sentences: {sum(y_sent):,} ({100*np.mean(y_sent):.1f}%)')

Building sentence-level dataset...
Sentences total: 89,271
Prop sentences: 15,957 (17.9%)


In [30]:
print('Building span-level dataset ...')
span_texts, y_tech, TC_CLASSES = build_span_dataset(df, min_count=5)
print(f'Span samples: {len(span_texts):,}')
print(f'Technique classes: {TC_CLASSES}')

Building span-level dataset ...
Span samples: 23,786
Technique classes: ['Appeal_to_Authority', 'Appeal_to_fear-prejudice', 'Bandwagon_Reductio_ad_hitlerum', 'Black-and-White_Fallacy', 'Causal_Oversimplification', 'Doubt', 'Exaggeration_Minimisation', 'Flag-Waving', 'Loaded_Language', 'Name_Calling_Labeling', 'Repetition', 'Slogans', 'Thought-terminating_Cliches', 'Whataboutism_Straw_Men_Red_Herring']


In [31]:
#Train/test split
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f'Train docs: {len(train_df):,} | Test docs: {len(test_df):,}')

Train docs: 3,604 | Test docs: 902


In [33]:
si_train_sents, si_train_y = build_sentence_dataset(train_df)
si_test_sents,  si_test_y  = build_sentence_dataset(test_df)

print(f'SI train: {len(si_train_sents):,} sentences, {np.mean(si_train_y):.1%} prop')
print(f'SI test: {len(si_test_sents):,} sentences, {np.mean(si_test_y):.1%} prop')

SI train: 71,459 sentences, 17.6% prop
SI test: 17,812 sentences, 19.2% prop


In [34]:
tc_X_train, tc_X_test, tc_y_train, tc_y_test = train_test_split(
    span_texts, y_tech,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_tech,
)

print(f'TC train: {len(tc_X_train):,} spans')
print(f'TC test: {len(tc_X_test):,} spans')

TC train: 19,028 spans
TC test: 4,758 spans


In [35]:
#Model 1: Span Identifier
print('Vectorising SI features...')

si_word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=40_000,
    sublinear_tf=True,
    analyzer='word',
)
si_char_vec = TfidfVectorizer(
    ngram_range=(2, 5),
    max_features=30_000,
    sublinear_tf=True,
    analyzer='char_wb',
)

si_X_train = hstack([
    si_word_vec.fit_transform(si_train_sents),
    si_char_vec.fit_transform(si_train_sents),
])
si_X_test = hstack([
    si_word_vec.transform(si_test_sents),
    si_char_vec.transform(si_test_sents),
])

print(f'SI feature matrix shape: {si_X_train.shape}')

Vectorising SI features ...
SI feature matrix shape: (71459, 70000)


In [36]:
print('Training Span Identifier...')

si_clf = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=RANDOM_STATE,
)
si_clf.fit(si_X_train, si_train_y)

print('Done.')

Training Span Identifier ...
Done.


In [38]:
#Default threshold (0.5) may not be optimal for our imbalanced distribution.
#We sweep over thresholds and pick the one maximising F1 on the PROP class

si_proba = si_clf.predict_proba(si_X_test)[:, 1]

best_f1, best_threshold = 0.0, 0.5
for t in np.arange(0.20, 0.80, 0.01):
    preds = (si_proba >= t).astype(int)
    f1 = f1_score(si_test_y, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_threshold = f1, round(float(t), 2)

print(f'Best PROP F1: {best_f1:.4f} at threshold = {best_threshold}')

Best PROP F1: 0.4848 at threshold = 0.49


In [39]:
si_preds = (si_proba >= best_threshold).astype(int)

print(classification_report(
    si_test_y, si_preds,
    target_names=['Non-propaganda', 'Propaganda'],
    zero_division=0,
))

                precision    recall  f1-score   support

Non-propaganda       0.90      0.76      0.83     14400
    Propaganda       0.39      0.64      0.48      3412

      accuracy                           0.74     17812
     macro avg       0.64      0.70      0.66     17812
  weighted avg       0.80      0.74      0.76     17812



In [40]:
#Model 2: Technique Classifier
print('Vectorising TC features ...')

tc_word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=30_000,
    sublinear_tf=True,
    analyzer='word',
)
tc_char_vec = TfidfVectorizer(
    ngram_range=(2, 5),
    max_features=20_000,
    sublinear_tf=True,
    analyzer='char_wb',
)

tc_X_train_v = hstack([
    tc_word_vec.fit_transform(tc_X_train),
    tc_char_vec.fit_transform(tc_X_train),
])
tc_X_test_v = hstack([
    tc_word_vec.transform(tc_X_test),
    tc_char_vec.transform(tc_X_test),
])

print(f'TC feature matrix shape: {tc_X_train_v.shape}')

Vectorising TC features ...
TC feature matrix shape: (19028, 50000)


In [41]:
print('Training Technique Classifier ...')

tc_clf = SGDClassifier(
    loss='modified_huber',
    alpha=1e-4,
    max_iter=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
tc_clf.fit(tc_X_train_v, tc_y_train)

print('Done.')

Training Technique Classifier ...
Done.


In [42]:
tc_preds = tc_clf.predict(tc_X_test_v)

tc_macro   = f1_score(tc_y_test, tc_preds, average='macro',    zero_division=0)
tc_weighted = f1_score(tc_y_test, tc_preds, average='weighted', zero_division=0)

print(f'Macro F1: {tc_macro:.4f}')
print(f'Weighted F1: {tc_weighted:.4f}')
print(classification_report(
    tc_y_test, tc_preds,
    labels=TC_CLASSES,
    zero_division=0,
))

Macro F1   : 0.3821
Weighted F1: 0.3510

=== Technique Classifier — Test Set Report ===
                                    precision    recall  f1-score   support

               Appeal_to_Authority       0.36      0.45      0.40       370
          Appeal_to_fear-prejudice       0.42      0.50      0.46       292
    Bandwagon_Reductio_ad_hitlerum       0.34      0.23      0.28      1217
           Black-and-White_Fallacy       0.37      0.50      0.43        68
         Causal_Oversimplification       0.33      0.35      0.34        92
                             Doubt       0.42      0.44      0.43       308
         Exaggeration_Minimisation       0.37      0.44      0.40       238
                       Flag-Waving       0.42      0.47      0.44       451
                   Loaded_Language       0.39      0.36      0.37       564
             Name_Calling_Labeling       0.46      0.52      0.49       300
                        Repetition       0.37      0.50      0.42       219

In [43]:
#Save models
si_bundle = {
    'word_vectorizer'  : si_word_vec,
    'char_vectorizer'  : si_char_vec,
    'classifier'       : si_clf,
    'threshold'        : best_threshold,
    'description'      : 'Sentence-level propaganda span identifier. '
                         'Input: sentence string. Output: P(propaganda).'
}

tc_bundle = {
    'word_vectorizer'  : tc_word_vec,
    'char_vectorizer'  : tc_char_vec,
    'classifier'       : tc_clf,
    'classes'          : TC_CLASSES,
    'description'      : 'Span-level technique classifier. '
                         'Input: span text string. Output: technique label.'
}

joblib.dump(si_bundle, SI_PATH)
joblib.dump(tc_bundle, TC_PATH)

print(f'SI model saved → {SI_PATH}')
print(f'TC model saved → {TC_PATH}')

SI model saved → /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_si.joblib
TC model saved → /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/distilled_tc.joblib


In [ ]:
#How to load the models and run inference on a new article
# ── Load from disk (simulates downstream usage) ───────────────────────────────
si = joblib.load(SI_PATH)
tc = joblib.load(TC_PATH)


def predict_propaganda(article_text: str,
                        si_bundle: dict,
                        tc_bundle: dict) -> list[dict]:
    """Run the full propaganda detection pipeline on a raw article.

    Steps
    -----
    1. Split article into sentences.
    2. Score each sentence with the Span Identifier.
    3. For sentences above the detection threshold, classify the technique.

    Returns
    -------
    List of dicts, each with keys:
        'sentence'  : str   — the flagged sentence
        'span'      : [int, int]  — [start_char, end_char] in the article
        'technique' : str   — predicted propaganda technique
        'si_score'  : float — span identifier confidence (0–1)
    """
    sents = split_sentences(article_text)
    if not sents:
        return []

    sent_strings = [s for s, _, _ in sents]

    #Identify spans
    si_feats = hstack([
        si_bundle['word_vectorizer'].transform(sent_strings),
        si_bundle['char_vectorizer'].transform(sent_strings),
    ])
    si_scores  = si_bundle['classifier'].predict_proba(si_feats)[:, 1]
    threshold  = si_bundle['threshold']
    is_prop    = si_scores >= threshold

    flagged = [
        (sent_strings[i], sents[i][1], sents[i][2], float(si_scores[i]))
        for i in range(len(sents))
        if is_prop[i]
    ]

    if not flagged:
        return []

    #Classify techniques
    flagged_texts = [f[0] for f in flagged]
    tc_feats = hstack([
        tc_bundle['word_vectorizer'].transform(flagged_texts),
        tc_bundle['char_vectorizer'].transform(flagged_texts),
    ])
    techniques = tc_bundle['classifier'].predict(tc_feats)

    results = []
    for (sent, ss, se, score), tech in zip(flagged, techniques):
        results.append({
            'sentence' : sent,
            'span'     : [ss, se],
            'technique': tech,
            'si_score' : round(score, 3),
        })

    return results

In [47]:
#Example
DEMO_ARTICLE = """
The treacherous elites are draining our nation dry while honest, hardworking
patriots watch helplessly from the sidelines. Either you stand with us or you
stand against everything we hold dear. The radical agenda pushed by these
globalist puppets must be exposed for what it truly is. Meanwhile, scientists
released a new study on agricultural yields in Southeast Asia. The heroic
resistance fighters continue their noble struggle for freedom and justice.
"""

results = predict_propaganda(DEMO_ARTICLE, si, tc)

print(f'Detected {len(results)} propaganda span(s):\n')
for r in results:
    print(f"• [{r['technique']}] (score={r['si_score']})")
    print(f"  Sentence: {r['sentence'][:120]}")

Detected 7 propaganda span(s):

• [Name_Calling_Labeling] (score=0.979)
  Sentence: The treacherous elites are draining our nation dry while honest, hardworking
• [Black-and-White_Fallacy] (score=0.767)
  Sentence: Either you stand with us or you
• [Flag-Waving] (score=0.709)
  Sentence: stand against everything we hold dear.
• [Appeal_to_Authority] (score=0.622)
  Sentence: The radical agenda pushed by these
• [Flag-Waving] (score=0.954)
  Sentence: globalist puppets must be exposed for what it truly is.
• [Appeal_to_Authority] (score=0.863)
  Sentence: The heroic
• [Flag-Waving] (score=0.601)
  Sentence: resistance fighters continue their noble struggle for freedom and justice.
